In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from sklearn.neural_network import MLPRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, Matern
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from scipy.interpolate import griddata, Rbf, interp2d
from scipy.optimize import curve_fit
import warnings
warnings.filterwarnings('ignore')

class AerodynamicSurfaceFitter:
    def __init__(self, excel_file='re_dataset.xlsx'):
        """
        Initialize the surface fitter with data from Excel file
        Expected columns: Re, Cl, alpha
        """
        self.data = pd.read_excel(excel_file)
        self.prepare_data()
        
    def prepare_data(self):
        """Prepare and validate the data"""
        print("Data shape:", self.data.shape)
        print("Columns:", self.data.columns.tolist())
        print("Data preview:")
        print(self.data.head())
        print("\nData ranges:")
        print(f"Reynolds number: {self.data['Re'].min():.0f} to {self.data['Re'].max():.0f}")
        print(f"Alpha (degrees): {self.data['alpha'].min():.1f} to {self.data['alpha'].max():.1f}")
        print(f"Cl: {self.data['Cl'].min():.3f} to {self.data['Cl'].max():.3f}")
        
        # Prepare features (X) and target (y)
        self.X = self.data[['Re', 'alpha']].values
        self.y = self.data['Cl'].values
        
        # Create train-test split
        self.X_train, self.X_test, self.y_train, self.y_test = train_test_split(
            self.X, self.y, test_size=0.2, random_state=42
        )
        
        # Scale features for neural networks and SVR
        self.scaler = StandardScaler()
        self.X_train_scaled = self.scaler.fit_transform(self.X_train)
        self.X_test_scaled = self.scaler.transform(self.X_test)
        
    def fit_random_forest(self):
        """Random Forest Regression"""
        print("\n=== Random Forest ===")
        rf = RandomForestRegressor(n_estimators=100, random_state=42, n_jobs=-1)
        rf.fit(self.X_train, self.y_train)
        y_pred = rf.predict(self.X_test)
        
        mse = mean_squared_error(self.y_test, y_pred)
        r2 = r2_score(self.y_test, y_pred)
        print(f"MSE: {mse:.6f}")
        print(f"R²: {r2:.6f}")
        
        return rf, {'mse': mse, 'r2': r2, 'name': 'Random Forest'}
    
    def fit_neural_network(self):
        """Multi-layer Perceptron Neural Network"""
        print("\n=== Neural Network ===")
        nn = MLPRegressor(hidden_layer_sizes=(100, 50, 25), max_iter=1000, 
                         random_state=42, alpha=0.01)
        nn.fit(self.X_train_scaled, self.y_train)
        y_pred = nn.predict(self.X_test_scaled)
        
        mse = mean_squared_error(self.y_test, y_pred)
        r2 = r2_score(self.y_test, y_pred)
        print(f"MSE: {mse:.6f}")
        print(f"R²: {r2:.6f}")
        
        return nn, {'mse': mse, 'r2': r2, 'name': 'Neural Network', 'scaled': True}
    
    def fit_svr(self):
        """Support Vector Regression"""
        print("\n=== Support Vector Regression ===")
        svr = SVR(kernel='rbf', C=100, gamma='scale', epsilon=0.01)
        svr.fit(self.X_train_scaled, self.y_train)
        y_pred = svr.predict(self.X_test_scaled)
        
        mse = mean_squared_error(self.y_test, y_pred)
        r2 = r2_score(self.y_test, y_pred)
        print(f"MSE: {mse:.6f}")
        print(f"R²: {r2:.6f}")
        
        return svr, {'mse': mse, 'r2': r2, 'name': 'SVR', 'scaled': True}
    
    def fit_gaussian_process(self):
        """Gaussian Process Regression"""
        print("\n=== Gaussian Process ===")
        kernel = RBF(length_scale=1.0) + Matern(length_scale=1.0, nu=2.5)
        gp = GaussianProcessRegressor(kernel=kernel, random_state=42, alpha=1e-6)
        gp.fit(self.X_train_scaled, self.y_train)
        y_pred = gp.predict(self.X_test_scaled)
        
        mse = mean_squared_error(self.y_test, y_pred)
        r2 = r2_score(self.y_test, y_pred)
        print(f"MSE: {mse:.6f}")
        print(f"R²: {r2:.6f}")
        
        return gp, {'mse': mse, 'r2': r2, 'name': 'Gaussian Process', 'scaled': True}
    
    def fit_polynomial(self, degree=3):
        """Polynomial regression"""
        print(f"\n=== Polynomial Regression (degree {degree}) ===")
        
        def poly_func(X, *coeffs):
            Re, alpha = X
            # Create polynomial features
            result = coeffs[0]  # constant term
            idx = 1
            for i in range(1, degree + 1):
                for j in range(i + 1):
                    result += coeffs[idx] * (Re**(i-j)) * (alpha**j)
                    idx += 1
            return result
        
        # Calculate number of coefficients needed
        n_coeffs = sum(range(2, degree + 3))
        
        try:
            # Fit polynomial
            popt, _ = curve_fit(poly_func, self.X_train.T, self.y_train, 
                              p0=np.ones(n_coeffs), maxfev=5000)
            
            # Make predictions
            y_pred = poly_func(self.X_test.T, *popt)
            
            mse = mean_squared_error(self.y_test, y_pred)
            r2 = r2_score(self.y_test, y_pred)
            print(f"MSE: {mse:.6f}")
            print(f"R²: {r2:.6f}")
            
            return (poly_func, popt), {'mse': mse, 'r2': r2, 'name': f'Polynomial (deg {degree})'}
        except:
            print("Polynomial fitting failed")
            return None, {'mse': float('inf'), 'r2': -float('inf'), 'name': f'Polynomial (deg {degree})'}
    
    def fit_rbf_interpolation(self):
        """Radial Basis Function interpolation"""
        print("\n=== RBF Interpolation ===")
        rbf = Rbf(self.X_train[:, 0], self.X_train[:, 1], self.y_train, 
                  function='multiquadric', smooth=0.1)
        
        y_pred = rbf(self.X_test[:, 0], self.X_test[:, 1])
        
        mse = mean_squared_error(self.y_test, y_pred)
        r2 = r2_score(self.y_test, y_pred)
        print(f"MSE: {mse:.6f}")
        print(f"R²: {r2:.6f}")
        
        return rbf, {'mse': mse, 'r2': r2, 'name': 'RBF Interpolation'}
    
    def compare_all_methods(self):
        """Fit all methods and compare results"""
        print("Fitting all methods...")
        
        methods = []
        results = []
        
        # Machine Learning methods
        rf, rf_result = self.fit_random_forest()
        methods.append(rf)
        results.append(rf_result)
        
        nn, nn_result = self.fit_neural_network()
        methods.append(nn)
        results.append(nn_result)
        
        svr, svr_result = self.fit_svr()
        methods.append(svr)
        results.append(svr_result)
        
        gp, gp_result = self.fit_gaussian_process()
        methods.append(gp)
        results.append(gp_result)
        
        # Classical methods
        poly2, poly2_result = self.fit_polynomial(degree=2)
        methods.append(poly2)
        results.append(poly2_result)
        
        poly3, poly3_result = self.fit_polynomial(degree=3)
        methods.append(poly3)
        results.append(poly3_result)
        
        rbf, rbf_result = self.fit_rbf_interpolation()
        methods.append(rbf)
        results.append(rbf_result)
        
        # Summary
        print("\n" + "="*60)
        print("COMPARISON SUMMARY")
        print("="*60)
        
        # Sort by R² score
        sorted_results = sorted(results, key=lambda x: x['r2'], reverse=True)
        
        print(f"{'Method':<25} {'R²':<10} {'MSE':<12}")
        print("-" * 50)
        for result in sorted_results:
            print(f"{result['name']:<25} {result['r2']:<10.4f} {result['mse']:<12.6f}")
        
        return methods, results
    
    def plot_surface(self, model, model_info, grid_size=50):
        """Plot 3D surface for a given model"""
        # Create grid for surface
        re_range = np.linspace(self.data['Re'].min(), self.data['Re'].max(), grid_size)
        alpha_range = np.linspace(self.data['alpha'].min(), self.data['alpha'].max(), grid_size)
        Re_grid, Alpha_grid = np.meshgrid(re_range, alpha_range)
        
        # Prepare grid points for prediction
        grid_points = np.column_stack([Re_grid.ravel(), Alpha_grid.ravel()])
        
        # Make predictions based on model type
        if model_info['name'] == 'Random Forest':
            Cl_pred = model.predict(grid_points)
        elif model_info.get('scaled', False):
            grid_points_scaled = self.scaler.transform(grid_points)
            Cl_pred = model.predict(grid_points_scaled)
        elif 'Polynomial' in model_info['name']:
            poly_func, popt = model
            Cl_pred = poly_func(grid_points.T, *popt)
        elif model_info['name'] == 'RBF Interpolation':
            Cl_pred = model(grid_points[:, 0], grid_points[:, 1])
        
        Cl_grid = Cl_pred.reshape(Re_grid.shape)
        
        # Create 3D plot
        fig = plt.figure(figsize=(12, 8))
        ax = fig.add_subplot(111, projection='3d')
        
        # Plot surface
        surf = ax.plot_surface(Re_grid, Alpha_grid, Cl_grid, cmap='viridis', 
                              alpha=0.8, edgecolor='none')
        
        # Plot original data points
        ax.scatter(self.data['Re'], self.data['alpha'], self.data['Cl'], 
                  c='red', s=20, alpha=0.6, label='Data points')
        
        ax.set_xlabel('Reynolds Number')
        ax.set_ylabel('Angle of Attack (degrees)')
        ax.set_zlabel('Coefficient of Lift (Cl)')
        ax.set_title(f'Cl Surface - {model_info["name"]} (R² = {model_info["r2"]:.4f})')
        
        # Add colorbar
        fig.colorbar(surf, shrink=0.5, aspect=5)
        ax.legend()
        
        plt.tight_layout()
        plt.show()
        
        return fig
    
    def plot_best_surfaces(self, top_n=3):
        """Plot surfaces for the top N performing methods"""
        methods, results = self.compare_all_methods()
        
        # Sort by R² score and take top N
        sorted_indices = sorted(range(len(results)), 
                              key=lambda i: results[i]['r2'], reverse=True)
        
        print(f"\nPlotting surfaces for top {top_n} methods...")
        
        for i in range(min(top_n, len(sorted_indices))):
            idx = sorted_indices[i]
            if results[idx]['r2'] > -float('inf'):  # Skip failed methods
                print(f"\nPlotting {results[idx]['name']}...")
                self.plot_surface(methods[idx], results[idx])

# Usage example
if __name__ == "__main__":
    # Create the fitter object
    fitter = AerodynamicSurfaceFitter('re_dataset.xlsx')
    
    # Compare all methods
    methods, results = fitter.compare_all_methods()
    
    # Plot the best performing surfaces
    fitter.plot_best_surfaces(top_n=3)
    
    # You can also plot a specific method:
    # best_idx = np.argmax([r['r2'] for r in results])
    # fitter.plot_surface(methods[best_idx], results[best_idx])

FileNotFoundError: [Errno 2] No such file or directory: 're_dataset.xlsx'